# Análise de Dados - Workforce Balance - Certificados

Este notebook tem como objetivo fazermos uma análise exploratória básica nos dados de forma a identificar quais transformações que teremos que fazer na pipeline. Em cada um dos csvs vamos verificar:

- Se os dados estão válidos
- Se os dados estão nulos
- Se há dados repetidos
- Se os tipos de dados são consistentes
- Se os dados estão disponíveis no tempo adequado

## Dados de colaboradores

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Users\conta\OneDrive\Documentos\GitHub\dbt_workforce_balance\csvs\Dados OSPA - Colaboradores.csv")
df.head()

,Id do colaborador,CPF do colaborador,Nome do colaborador,Cidade de atuação colaborador,Id da equipe do colaborador,Data da carga
0,988001,10098057601,Ana Clara Silva,São Paulo,189576,26/09/2026
1,988002,10098057602,Bruno Oliveira Santos,São Caetano,189576,26/09/2026
2,988003,10098057603,Carlos Eduardo Souza,Santo André,189577,26/09/2026
3,988004,10098057604,Daniela Almeida Lima,São Paulo,189577,26/09/2026
4,988005,10098057605,Eduardo Costa Pereira,São Caetano,189578,26/09/2026


### Acurácia

In [3]:
def validar_cpf(cpf: pd.Series) -> pd.Series:
    cpf = (
        cpf.astype("string")
        .str.replace(r"\D", "", regex=True)
    )

    repetidos = [str(i) * 11 for i in range(10)]

    # CPF precisa ter 11 dígitos e não pode ser uma sequência repetida
    valido = (
        cpf.str.len().eq(11) &
        ~cpf.isin(repetidos)
    )

    # Primeiro dígito verificador
    numeros = cpf.str.slice(0,9)
    soma = sum(
        numeros.str[i].astype("Int64") * (10-i)
        for i in range(9)
    )
    digito1 = ((soma * 10) % 11).replace(10,0)
    valido &= digito1.eq(cpf.str[9].astype("Int64"))

    # Segundo dígito verificador
    numeros = cpf.str.slice(0,10)
    soma = sum(
        numeros.str[i].astype("Int64") * (11 - i)
        for i in range(10)
    )

    digito2 = ((soma * 10)%11).replace(10,0)

    valido &= digito2.eq(cpf.str[10].astype("Int64"))

    return valido.fillna(False)

In [4]:
df["CPF Validado"] = validar_cpf(df["CPF do colaborador"])

In [5]:
df["CPF Validado"].value_counts()

CPF Validado
False    98
True      1
Name: count, dtype: Int64

### Completude

In [6]:
df.isnull().sum()

Id do colaborador                0
CPF do colaborador               0
Nome do colaborador              0
Cidade de atuação colaborador    0
Id da equipe do colaborador      0
Data da carga                    0
CPF Validado                     0
dtype: int64

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99 entries, 0 to 98
Data columns (total 7 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Id do colaborador              99 non-null     int64  
 1   CPF do colaborador             99 non-null     int64  
 2   Nome do colaborador            99 non-null     str    
 3   Cidade de atuação colaborador  99 non-null     str    
 4   Id da equipe do colaborador    99 non-null     int64  
 5   Data da carga                  99 non-null     str    
 6   CPF Validado                   99 non-null     boolean
dtypes: boolean(1), int64(3), str(3)
memory usage: 9.0 KB


In [8]:
df["CPF do colaborador"] = df["CPF do colaborador"].astype("str")
df["Id do colaborador"] = df["Id do colaborador"].astype("str")
df["Data da carga - date"] = pd.to_datetime(df["Data da carga"], format = "%d/%m/%Y")

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99 entries, 0 to 98
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Id do colaborador              99 non-null     str           
 1   CPF do colaborador             99 non-null     str           
 2   Nome do colaborador            99 non-null     str           
 3   Cidade de atuação colaborador  99 non-null     str           
 4   Id da equipe do colaborador    99 non-null     int64         
 5   Data da carga                  99 non-null     str           
 6   CPF Validado                   99 non-null     boolean       
 7   Data da carga - date           99 non-null     datetime64[us]
dtypes: boolean(1), datetime64[us](1), int64(1), str(5)
memory usage: 11.4 KB


### Unicidade

In [10]:
df.duplicated().value_counts()

False    99
Name: count, dtype: int64

In [11]:
df["CPF do colaborador"].duplicated().value_counts()

CPF do colaborador
False    99
Name: count, dtype: int64

In [12]:
df.to_csv(r"C:\Users\conta\OneDrive\Documentos\GitHub\dbt_workforce_balance\csvs\Dados OSPA - Colaboradores - verified.csv")

Mudanças necessárias

- Necessário ver com as áreas de negócio o que fazer com CPFs não batendo.